# Load CARWatch logs and Study Manager results

This notebook demonstrates the package capabilities available after the raw-log and Study Manager loaders. It uses synthetic data from `examples/data` and does not access the ignored `playground` directory.

Register the project environment before opening the notebook with `uv run poe conf_jupyter`, then select the `carwatch` kernel.

In [1]:
from pathlib import Path

from carwatch.io import load_logs, load_study_results
#from carwatch.logs import extract_samples, extract_awakening
from carwatch.study_manager import extract_awakening, extract_samples

In [2]:
DATA_DIR = Path("examples/data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATA_DIR

PosixPath('data')

## Raw app logs

`load_logs` parses semicolon-delimited app logs, converts Unix milliseconds into timezone-aware timestamps, and retains JSON payloads as dictionaries.

In [3]:
logs = load_logs(
    DATA_DIR / "carwatch_demo_VP01_20250515.csv",
    tz="Europe/Berlin",
)
logs[["timestamp", "action", "source_file"]]

,timestamp,action,source_file
0,2025-05-15 06:13:30.799000+02:00,spontaneous_awakening,carwatch_demo_VP01_20250515.csv
1,2025-05-15 06:13:55.999000+02:00,barcode_scanned,carwatch_demo_VP01_20250515.csv
2,2025-05-15 06:43:51.091000+02:00,barcode_scanned,carwatch_demo_VP01_20250515.csv
3,2025-05-15 06:58:52+02:00,barcode_scanned,carwatch_demo_VP01_20250515.csv
4,2025-05-15 07:13:47+02:00,barcode_scanned,carwatch_demo_VP01_20250515.csv


## Extract sampling and awakening events

The extraction functions retain the expected sample, the physical tube that was scanned, and any mismatch between them.

In [4]:
log_samples = cw.logs.extract_sample_events_from_raw_logs(logs)
log_awakening = cw.logs.extract_awakening_events_from_raw_logs(logs)

log_samples[[
    "sampling_time",
    "sample",
    "sample_scanned",
    "barcode",
    "sample_mismatch",
]]

SchemaError: Study results require a 'participant' index.

In [5]:
log_awakening

NameError: name 'log_awakening' is not defined

## Study Manager export

`load_study_results` retains one row per participant. Its column levels identify the day, sample, and variable without repeating day-level information for every sample.

In [6]:
study_results = load_study_results(
    DATA_DIR / "study_results.csv",
    tz="Europe/Berlin",
)
study_results

day                                D1                            \
sample                            day                             
variable                         date            awakening_time   
participant                                                       
VP01        2025-05-15 00:00:00+02:00 2025-05-15 06:13:30+02:00   

day                                                                    \
sample                                                             B1   
variable    awakening_type mismatch_summary             sampling_time   
participant                                                             
VP01           self-report    B2->B3;B3->B2 2025-05-15 06:13:55+02:00   

day                                                                     \
sample                                                     B2            
variable     barcode sample_scanned             sampling_time  barcode   
participant                                                              
VP01         0010101             B1 2025-05-15 06:43:51+02:00  0010103   

day                                                                           \
sample                                            B3                           
variable    sample_scanned             sampling_time  barcode sample_scanned   
participant                                                                    
VP01                    B3 2025-05-15 06:58:52+02:00  0010102             B2   

day                                                            
sample                             B4                          
variable                sampling_time  barcode sample_scanned  
participant                                                    
VP01        2025-05-15 07:13:47+02:00  0010104             B4

## Focused Study Manager views

The helper functions extract day-level awakening information and sample-level observations only when those representations are needed. The original mismatch summary remains available once per day.

In [7]:
study_awakening = cw.logs.extract_awakening_events_from_summary(study_results)
study_days = cw.logs.extract_day_summary_from_summary(study_results)
study_samples = cw.logs.extract_sample_events_from_summary(study_results)
study_awakening

,,date,awakening_time,awakening_type,mismatch_summary
participant,day,,,,
VP01,D1,2025-05-15 00:00:00+02:00,2025-05-15 06:13:30+02:00,self-report,B2->B3;B3->B2


In [ ]:
study_days

In [ ]:
study_samples

## Audit recorded sample swaps

The per-sample mismatch flag is derived from the expected sample and `sample_scanned`.

In [9]:
mismatches = study_samples.loc[study_samples["sample_mismatch"].fillna(False)]
mismatches[["barcode", "sample_scanned"]]

barcode sample_scanned
participant day sample                        
VP01        D1  B2      0010103             B3
                B3      0010102             B2